# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
# logging.getLogger("py4j").setLevel(logging.WARNING)
# logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/data/cdr/teleparser/Semana100/claro/fixa_sigma")
files_claro = [str(file) for file in folder_claro.rglob("*.parquet") if file.is_file()]
files_claro[:5]

### Arquivos intermediário e final

In [ ]:
extracted_claro = "/assets/data/cdr/teletools/db/extraidos/sigma_claro_extraido.parquet"
transformed_claro = "/assets/data/cdr/teletools/db/transformados/sigma_claro_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
df_raw = spark.read.parquet(*files_claro)
df_raw.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
columns_to_keep = [col for col in df_raw.columns if 'type' in col] + [col for col in df_raw.columns if 'time' in col]
columns_to_keep

In [ ]:
df_raw.groupBy("extension_2_record_type").count().show()

In [ ]:
df_raw.filter(F.col("a_call_duration")=="------").select(columns_to_keep).show(5)

In [ ]:
df_raw.filter(F.col("a_incoming_seizure_time")=="").count()

In [ ]:
column_mapping = [
    ("source_file", "_arquivo_origem"),
    ("format", "_tipo_cdr"),
    ("a_exchange_id", "bilhetador"),
    ("a_calling_party", "numero_origem"),
    ("a_called_party", "numero_destino"),
    ("a_start_date", "_data"),
    ("a_start_time", "_hora"),
    ("a_duration_seconds", "duracao"),
    ("a_incoming_route_name", "rota_entrada"),
    ("a_outgoing_route_name", "rota_saida"),
    ("a_call_class", "resultado_chamada"),
    ("a_incoming_seizure_time", "_hora_nao_atendidas"),
    ("a_sequence_number", "referencia"),
]

fill_char = "-"

select_expr = []
for source_col, target_col in column_mapping:
    if source_col in df_raw.columns:
        source_expr = (
            F.col(f"`{source_col}`") if "." in source_col else F.col(source_col)
        )
    else:
        source_expr = F.lit(None).cast("string")
    if fill_char:
        # fill_char entra no padrão sem escape de metacaracteres.
        fill_pattern = f"{fill_char}+$"
        source_expr = F.regexp_replace(source_expr, fill_pattern, "")
        
    select_expr.append(source_expr.alias(target_col))
df = df_raw.select(*select_expr)
df.show(5,truncate=False)

In [ ]:
# df.filter(F.col("duracao").isNotNull()).show(5)

## Extração

In [ ]:
# %%time
# extractor = CDRParquetExtractor(spark)
# # 2. Silencia a saída padrão do Log4j do Spark para não duplicar no terminal
# jvm = spark.sparkContext._jvm
# log4j = jvm.org.apache.logging.log4j
# configurator = log4j.core.config.Configurator

# logger_names = (
#     "org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat",
#     "org.apache.spark.sql.execution.datasources.FileScanRDD",
# )
# previous_levels = {
#     name: log4j.LogManager.getLogger(name).getLevel()
#     for name in logger_names
# }

# try:
#     for name in logger_names:
#         configurator.setLevel(name, log4j.Level.ERROR)
        
#     extractor.extract(files_claro, extracted_claro, "stfc_sigma_claro")
# finally:
#     for name, level in previous_levels.items():
#         configurator.setLevel(name, level)

In [ ]:
%%time
extractor = CDRParquetExtractor(spark)
extractor.extract(files_claro, extracted_claro, "stfc_sigma_claro")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5)

## Transformação

In [ ]:
%%time
transformer = CDRTransformer(spark)
transformer.transform_stfc_sigma_claro(extracted_claro, transformed_claro)

In [ ]:
df = spark.read.parquet(transformed_claro)
df.show(5)

In [ ]:
df.limit(5).toPandas()

In [ ]:
df.groupBy("no_tipo_chamada").count().show()

In [ ]:
from teleutils.core.extractors.schemas import PARQUET_DEFAULT_SCHEMAS, TEXT_DEFAULT_SCHEMAS

In [ ]:
schemas_names = [x for x in PARQUET_DEFAULT_SCHEMAS] + [x for x in TEXT_DEFAULT_SCHEMAS]
schemas_names

In [ ]:
schemas_names.sort()

In [ ]:
for i,schema in enumerate(schemas_names):
    print(f"{schema}")